### 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 2: Load Cleaned Dataset

In [2]:
df = pd.read_csv("../data/processed/customer_churn_cleaned.csv")

df.head()

,Country,State,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,United States,California,Los Angeles,Male,No,No,No,2,Yes,No,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,United States,California,Los Angeles,Female,No,No,Yes,2,Yes,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,United States,California,Los Angeles,Female,No,No,Yes,8,Yes,Yes,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,United States,California,Los Angeles,Female,No,Yes,Yes,28,Yes,Yes,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003
4,United States,California,Los Angeles,Male,No,No,Yes,49,Yes,Yes,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340


### 3: Dataset Overview

In [3]:
df.shape

(7032, 24)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country            7032 non-null   object 
 1   State              7032 non-null   object 
 2   City               7032 non-null   object 
 3   Gender             7032 non-null   object 
 4   Senior Citizen     7032 non-null   object 
 5   Partner            7032 non-null   object 
 6   Dependents         7032 non-null   object 
 7   Tenure Months      7032 non-null   int64  
 8   Phone Service      7032 non-null   object 
 9   Multiple Lines     7032 non-null   object 
 10  Internet Service   7032 non-null   object 
 11  Online Security    7032 non-null   object 
 12  Online Backup      7032 non-null   object 
 13  Device Protection  7032 non-null   object 
 14  Tech Support       7032 non-null   object 
 15  Streaming TV       7032 non-null   object 
 16  Streaming Movies   7032 

In [5]:
df.columns.tolist()

['Country',
 'State',
 'City',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Value',
 'CLTV']

### 4: Separate Target Variable

In [7]:
X = df.drop("Churn Value", axis=1)

y = df["Churn Value"]

#### Verify

In [8]:
X.shape

(7032, 23)

In [9]:
y.shape

(7032,)

### 5: Identify Column Types

In [10]:
cat_cols = X.select_dtypes(include="object").columns.tolist()

num_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical Columns:")
print(cat_cols)

print("\nNumerical Columns:")
print(num_cols)

Categorical Columns:
['Country', 'State', 'City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical Columns:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']


### 6: Drop Weak/High-Cardinality Features

In [11]:
df.drop(
    columns=[
        "Country",
        "State",
        "City"
    ],
    inplace=True
)

#### 
Why?

- Country → same value
- State → same value
- City → too many categories
- They add complexity and little predictive power.

### 7: One-Hot Encoding

In [12]:
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

#### Check

In [13]:
df_encoded.shape

(7032, 32)

In [14]:
df_encoded.head()

,Tenure Months,Monthly Charges,Total Charges,Churn Value,CLTV,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,2,53.85,108.15,1,3239,True,False,False,False,True,...,False,False,False,False,False,False,True,False,False,True
1,2,70.70,151.65,1,2701,False,False,False,True,True,...,False,False,False,False,False,False,True,False,True,False
2,8,99.65,820.50,1,5372,False,False,False,True,True,...,False,True,False,True,False,False,True,False,True,False
3,28,104.80,3046.05,1,5003,False,False,True,True,True,...,False,True,False,True,False,False,True,False,True,False
4,49,103.70,5036.30,1,5340,True,False,False,True,True,...,False,True,False,True,False,False,True,False,False,False


### 8: Separate X and Y Again

In [15]:
X = df_encoded.drop("Churn Value", axis=1)

y = df_encoded["Churn Value"]

### 9: Train-Test Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### Check

In [17]:
print(X_train.shape)
print(X_test.shape)

(5625, 31)
(1407, 31)


### 10: Feature Scaling

#### Only scale numerical columns.

In [18]:
num_cols = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "CLTV"
]

#### Create scaler:

In [19]:
scaler = StandardScaler()

#### Fit on training data:

In [20]:
X_train[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)

#### Transform test data:

In [21]:
X_test[num_cols] = scaler.transform(
    X_test[num_cols]
)

### 11: Save Processed Data

In [24]:
X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

### 12: Save Scaler

In [23]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']